# SA T2 ACCESS-CM2 static — temporal (sequence-conditioned) downscaling

Companion to `SA_downscaling_refinement_T2_ACCESS-CM2_static.ipynb`. That notebook
covers the **frame-independent** Phase-1 + Phase-2 workflow; this one covers the
**temporal** extension on branch `Prithvi-UNet_temporal_model`.

The task is *causal, same-day, sequence-conditioned downscaling*: for output date
`t` the model uses the coarse predictors at `t` **and earlier**, with zero
predictor-to-target lead time. It is not a forecast.

Configs (identical except for `temporal.backend` and output paths):

- `examples/CORDEX_ML/SA_downscaling_refinement_T2_ACCESS-CM2_static_temporal_recurrent.yaml`
- `examples/CORDEX_ML/SA_downscaling_refinement_T2_ACCESS-CM2_static_temporal_mamba.yaml`

Every step below has an equivalent CLI invocation, shown in the cell above it, so
nothing here is notebook-only.

Run with the `Prithvi` environment:

```bash
mamba run -n Prithvi jupyter lab
```

In [ ]:
import json, os, sys, warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Run from the repository root.
while not os.path.isdir("granitewxc") and os.path.basename(os.getcwd()) != "granite-wxc":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

import numpy as np
import torch
import matplotlib.pyplot as plt

from granitewxc.utils.config import get_config
from granitewxc.temporal.config import parse_temporal_config
from granitewxc.temporal.backends import fused_mamba_available

CONFIG_R = "examples/CORDEX_ML/SA_downscaling_refinement_T2_ACCESS-CM2_static_temporal_recurrent.yaml"
CONFIG_M = "examples/CORDEX_ML/SA_downscaling_refinement_T2_ACCESS-CM2_static_temporal_mamba.yaml"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)
print("fused mamba_ssm available:", fused_mamba_available())

## 1. Audit the configuration before touching weights

Equivalent CLI:

```bash
mamba run -n Prithvi python examples/CORDEX_ML/cordex_temporal_training.py describe \
    --config examples/CORDEX_ML/SA_downscaling_refinement_T2_ACCESS-CM2_static_temporal_recurrent.yaml \
    --splits train validation test
```

This reports what the config *actually resolves to*: the real time axis (including
its discontinuities), how many contiguous runs and windows each split yields, and
the checkpoint contract. Note the 12 discontinuities in the SA training file — ten
2-day gaps where 29 February was removed, plus the 1980→2080 concatenation.

In [ ]:
config = get_config(CONFIG_R)
cfg = parse_temporal_config(config.temporal)

print("case          :", config.case_name)
print("targets       :", list(config.data.output_vars))
print("backend       :", cfg.backend)
print("mode          :", cfg.mode, "| causal:", cfg.causal, "| lead_time_days:", cfg.lead_time_days)
print("context/warmup/output/stride:",
      cfg.context_length, cfg.warmup_length, cfg.output_length, cfg.sequence_stride)
print("native n_input_timestamps   :", config.data.n_input_timestamps,
      "  <- a DIFFERENT axis from context_length")

In [ ]:
from granitewxc.temporal.sequence_dataset import TemporalSequenceDataset
from granitewxc.temporal.sources import build_frame_source
from granitewxc.temporal.training import _split_dates

for split in ("train", "validation", "test"):
    source = build_frame_source(config, split)
    start, end = _split_dates(config, split)
    ds = TemporalSequenceDataset(
        source, window_length=cfg.context_length,
        stride=cfg.sequence_stride if split == "train" else cfg.output_length,
        cadence_days=cfg.cadence_days,
        crop_size=(config.data.target_size_lat, config.data.target_size_lon),
        random_crop=False, static_channels=1, date_start=start, date_end=end,
    )
    d = ds.describe()
    print(f"{split:11s} {d['n_frames']:>5} frames  {d['n_runs']:>2} runs  "
          f"run_len {d['run_length_min']}-{d['run_length_max']}  "
          f"{d['n_windows']:>5} windows  steps={d['time_axis']['step_histogram']}")

## 2. Is the case event-aligned?

Day-by-day losses and event-paired metrics are only valid if predictors and
targets on the same date describe the same weather. This is checked, not assumed.

```bash
mamba run -n Prithvi python examples/CORDEX_ML/cordex_temporal_diagnostics.py \
    --config examples/CORDEX_ML/SA_downscaling_refinement_T2_ACCESS-CM2_static_temporal_recurrent.yaml \
    --days 3650 --output artifacts/sa_temporal_diagnostics.json
```

The script exits non-zero if any target fails, in which case
`temporal.evaluation.event_paired` must be set false and evaluation run with
`--not-event-aligned`.

In [ ]:
path = "artifacts/sa_temporal_diagnostics.json"
if os.path.isfile(path):
    diag = json.load(open(path))
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    for ax, (tname, entry) in zip(axes, diag["event_alignment"].items()):
        lags = sorted(int(k) for k in entry["lag_correlation"])
        vals = [entry["lag_correlation"][str(l)] for l in lags]
        ax.axvline(0, color="0.8", lw=1)
        ax.plot(lags, vals, "o-")
        ax.set_title(f"{tname}  (probe {entry['probe_predictor']})\n"
                     f"peak lag {entry['peak_lag']:+d}, "
                     f"{'ALIGNED' if entry['event_aligned'] else 'NOT ALIGNED'}")
        ax.set_xlabel("predictor lag (days)"); ax.set_ylabel("anomaly correlation")
    plt.tight_layout(); plt.show()

    print("History headroom (in-sample linear R2, deseasonalized, local):")
    for tname, h in diag["history_headroom"].items():
        print(f"  {tname:8s} same-day {h['r2_same_day_only']:.4f}  "
              f"+lags1-3 {h['gain_lags_1_3']:+.4f}  "
              f"(residual variance -{100*h['residual_variance_reduction_lags_1_3']:.1f}%)")
    print("\nTarget persistence (deseasonalized domain mean):")
    for tname, p in diag["target_persistence_deseasonalized"].items():
        print(f"  {tname:8s} " + "  ".join(f"{k}={v:.3f}" for k, v in p.items() if k.startswith("lag")))
else:
    print(f"{path} not found -- run the diagnostics command above first.")

## 3. Engineering checks on real data

```bash
mamba run -n Prithvi python examples/CORDEX_ML/cordex_temporal_training.py check \
    --config examples/CORDEX_ML/SA_downscaling_refinement_T2_ACCESS-CM2_static_temporal_recurrent.yaml \
    --split validation --output artifacts/temporal_check_SA_recurrent.json
```

Six checks, each of which fails for a specific reason:

| check | requirement |
|---|---|
| checkpoint migration | 208 spatial tensors load; only `temporal_adapter.*` may be missing |
| legacy parity (gate=0) | emitted field **bit-for-bit** equals the frame-independent prediction |
| near-identity (default gate) | perturbation ~1e-5 relative |
| gradient flow | **zero** temporal parameters with zero gradient |
| causality | perturbing the last frame leaves earlier outputs bit-for-bit unchanged |
| chunk consistency | chunked inference **exactly** equals a single pass |

In [ ]:
for backend, path in (("recurrent", "artifacts/temporal_check_SA_recurrent.json"),
                      ("mamba", "artifacts/temporal_check_SA_mamba.json")):
    if not os.path.isfile(path):
        print(f"{backend}: {path} not found -- run the check command above.")
        continue
    d = json.load(open(path)); m = d["migration"]
    print(f"--- {backend} ---")
    print(f"  migration        : loaded ok, missing_other={len(m['missing_other'])}, "
          f"unexpected={len(m['unexpected'])}, missing_temporal={len(m['missing_temporal'])}")
    print(f"  temporal params  : {d['temporal_parameters']:,}")
    print(f"  legacy parity    : identical={d['legacy_parity_gate_zero']['bitwise_identical']}, "
          f"maxdiff={d['legacy_parity_gate_zero']['max_abs_diff']}")
    print(f"  near-identity    : rel dev={d['near_identity_default_gate']['mean_relative_deviation']:.2e}")
    print(f"  zero-grad params : {d['gradient_flow']['n_zero_gradient']}/{d['gradient_flow']['n_temporal_tensors']}")
    print(f"  causal           : {d['causality']['earlier_frames_bitwise_unchanged']}")
    print(f"  chunk exact      : {d['chunk_consistency']['exact']} "
          f"(maxdiff {d['chunk_consistency']['max_abs_diff_overall']})")

## 4. Fine-tune

The adapter starts near-identity, so training departs from the frame-independent
model rather than from noise. The backbone stays frozen until epoch 3.

```bash
# recurrent (ConvGRU)
mamba run -n Prithvi python examples/CORDEX_ML/cordex_temporal_training.py train \
    --config examples/CORDEX_ML/SA_downscaling_refinement_T2_ACCESS-CM2_static_temporal_recurrent.yaml

# temporal Mamba
mamba run -n Prithvi python examples/CORDEX_ML/cordex_temporal_training.py train \
    --config examples/CORDEX_ML/SA_downscaling_refinement_T2_ACCESS-CM2_static_temporal_mamba.yaml

# bounded smoke run
mamba run -n Prithvi python examples/CORDEX_ML/cordex_temporal_training.py train \
    --config ...recurrent.yaml --max-steps 50 --max-val-steps 10 --epochs 1
```

`--max-steps` caps optimizer steps per epoch, which is how the bounded comparison
in `docs/temporal_model_results.md` was run.

In [ ]:
# Uncomment to train from the notebook. ~4.3 s per 7-frame window on an RTX PRO 6000,
# ~17 GiB peak, so a full epoch over 1163 windows is ~80 minutes.
#
# from granitewxc.temporal.training import train_temporal_model
# summary = train_temporal_model(
#     config, cfg, device=DEVICE,
#     output_dir="examples/CORDEX_ML/runs_temporal/notebook_recurrent/checkpoints",
#     max_steps_per_epoch=50, max_val_steps=10, num_epochs=1, batch_size=1,
# )
# print(json.dumps(summary["history"], indent=2, default=str))
print("training cell is commented out by default; uncomment to run")

## 5. The full bounded comparison

Five variants sharing data, splits, seed, step count, learning rates and per-frame
loss, differing only in how time is handled:

| variant | temporal pathway | isolates |
|---|---|---|
| `baseline` | gate 0, untrained | the existing frame-independent model |
| `spatial_ft` | inert (gate 0, temporal frozen) | does *any* further fine-tuning help? |
| `time_only` | full module, state zeroed every frame | date conditioning **without** memory |
| `convgru` | full recurrent | |
| `mamba` | full SSD | |

Acceptance tolerances are fixed in the script **before** any test result is read.

```bash
mamba run -n Prithvi python examples/CORDEX_ML/cordex_temporal_experiment.py \
    --out examples/CORDEX_ML/runs_temporal/experiment \
    --steps 600 --val-steps 60 --epochs 1 --test-years 3
```

In [ ]:
root = "examples/CORDEX_ML/runs_temporal/experiment"
sc_path = os.path.join(root, "scorecard.json")
if os.path.isfile(sc_path):
    sc = json.load(open(sc_path))
    for name, entry in sc["variants"].items():
        verdict = {True: "ACCEPTED", False: "NOT ACCEPTED", None: "control"}[entry.get("scientific_acceptance")]
        print(f"\n{name}: {verdict}  primary={entry['pass_primary']} guardrail={entry['pass_guardrail']}")
        for var, detail in entry["variables"].items():
            for group in ("primary", "guardrail"):
                for lab, v in detail[group].items():
                    if isinstance(v, dict) and v.get("status") == "FAIL":
                        print(f"    FAIL {var}/{lab}: baseline={v['baseline']:.5g} variant={v['variant']:.5g}")
else:
    print(f"{sc_path} not found -- run the experiment command above.")

## 6. Inference and evaluation

```bash
mamba run -n Prithvi python examples/CORDEX_ML/cordex_temporal_training.py infer \
    --config ...recurrent.yaml --split test \
    --checkpoint examples/CORDEX_ML/runs_temporal/SA_T2_ACCESS-CM2_static_temporal_recurrent/checkpoints/best.ckpt

mamba run -n Prithvi python examples/CORDEX_ML/cordex_temporal_training.py evaluate \
    --config ...recurrent.yaml --predictions <path printed by infer>
```

For the **free-running climate-change application** (2041–2060 / 2080–2099), pass
`--not-event-aligned`: those periods are a different climate state and date-paired
scores against a single realization would be meaningless.

In [ ]:
# Plot a prediction/target pair and the domain-mean trajectory from the experiment.
npz = os.path.join(root, "convgru", "predictions.npz")
if os.path.isfile(npz):
    d = np.load(npz, allow_pickle=False)
    pred, targ = d["pred"], d["target"]
    names = [str(v) for v in d["output_vars"]]
    day = 30
    fig, axes = plt.subplots(len(names), 3, figsize=(12, 3.4 * len(names)))
    axes = np.atleast_2d(axes)
    for i, nm in enumerate(names):
        vmin, vmax = np.nanpercentile(targ[day, i], [2, 98])
        for j, (arr, ttl) in enumerate(((targ[day, i], "target"), (pred[day, i], "temporal model"))):
            im = axes[i, j].imshow(arr, origin="lower", vmin=vmin, vmax=vmax)
            axes[i, j].set_title(f"{nm} {ttl}"); plt.colorbar(im, ax=axes[i, j], shrink=0.8)
        diff = pred[day, i] - targ[day, i]
        lim = float(np.nanpercentile(np.abs(diff), 98))
        im = axes[i, 2].imshow(diff, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
        axes[i, 2].set_title(f"{nm} error"); plt.colorbar(im, ax=axes[i, 2], shrink=0.8)
    plt.tight_layout(); plt.show()

    fig, axes = plt.subplots(len(names), 1, figsize=(11, 2.8 * len(names)))
    axes = np.atleast_1d(axes)
    for i, nm in enumerate(names):
        axes[i].plot(np.nanmean(targ[:120, i], axis=(1, 2)), label="target", lw=1.4)
        axes[i].plot(np.nanmean(pred[:120, i], axis=(1, 2)), label="temporal", lw=1.0)
        axes[i].set_title(f"{nm}: domain-mean trajectory, first 120 days of the test period")
        axes[i].legend()
    plt.tight_layout(); plt.show()
else:
    print(f"{npz} not found -- run the experiment first.")

## 7. Stochastic refinement

The temporal deterministic model is a usable baseline on its own; refinement is
**not** a prerequisite. All four existing refiners work unchanged.

`temporal.refinement.temporal_conditioning: none` (the shipped default) keeps the
existing per-date Phase-2 path byte-identical, so existing Phase-2 checkpoints stay
loadable. Setting `time_features` or `latent_state` widens the refiner's
conditioning and therefore **requires retraining Phase 2** — the code raises with
the exact channel counts rather than loading mismatched weights.

In [ ]:
from granitewxc.temporal.refinement import (
    AR1NoiseSource, check_refiner_temporal_compatibility,
    RefinerTemporalCompatibilityError, temporal_conditioning_channels,
)

print("extra conditioning channels by mode:")
for mode in ("none", "time_features", "latent_state"):
    import dataclasses
    r = dataclasses.replace(cfg.refinement, temporal_conditioning=mode)
    print(f"  {mode:14s} -> +{temporal_conditioning_channels(r, time_feature_dim=5)}")

try:
    check_refiner_temporal_compatibility(
        checkpoint_cond_channels=20, required_cond_channels=25,
        temporal_conditioning="time_features", checkpoint_path="existing_phase2.ckpt")
except RefinerTemporalCompatibilityError as exc:
    print("\nIncompatible Phase-2 checkpoint correctly rejected:\n ", str(exc)[:300], "...")

# AR(1) noise keeps every frame's marginal exactly N(0,1); only the between-frame
# correlation changes, so the refiner still sees the distribution it was trained on.
gens = [torch.Generator().manual_seed(1000 + m) for m in range(4)]
for rho in (0.0, 0.7):
    src = AR1NoiseSource(gens, batch_size=1, rho=rho)
    traj = np.stack([src.frame_noise((4, 1, 8, 8), "cpu", torch.float32).numpy().reshape(4, -1)
                     for _ in range(200)])
    lag1 = np.mean([np.corrcoef(traj[1:, m].ravel(), traj[:-1, m].ravel())[0, 1] for m in range(4)])
    print(f"rho={rho}: measured lag-1 noise autocorr={lag1:+.3f}, "
          f"per-frame std={traj.std():.3f} (target 1.000)")

## 8. Limitations

- The bounded run in `docs/temporal_model_results.md` is a few hundred optimizer
  steps on one GPU, not a converged experiment. Read the scorecard verdict, not
  the training loss.
- Scientific acceptance requires improved temporal/event representation **and**
  beating both the `spatial_ft` and `time_only` controls **and** no material
  spatial degradation. If the scorecard says NOT ACCEPTED, the checkpoints are
  experimental — the exact commands to continue are in the results document.
- The Phase-1 scalers were fitted over the whole 1961–1980 + 2080–2099 record, so
  they saw the validation window. They cannot be recomputed without invalidating
  the Phase-1 checkpoint, and the effect is identical across all variants, so the
  *comparison* is unaffected while absolute validation numbers are mildly
  optimistic.
- `mamba_ssm`'s fused kernels are not installable in this Windows/CUDA
  environment, so the Mamba backend runs the in-repo pure-PyTorch SSD recurrence
  — same mathematics and parameters, slower.
- The NARR/PRISM temporal configs are schema-complete and fully validated, but the
  NARR/PRISM archives and the `narr_prism_California` Phase-1 checkpoint are not
  present on this machine, so that case has **not** been executed end-to-end.